# **Impacto da Poluição Atmosférica na Saúde Respiratória: Uma Análise de Regressão com Dados Globais**

## *Analisando a correlação entre emissões de PM2.5 e taxas de internação por doenças respiratórias*

**Autores:** André Vinicius Moura Koraleski, Marcelo Soares, Raphael Alves de Lima Soares

**Instituição:** Universidade Federal de Goiás

**Disciplina:** Probabilidade e Estatística

**Data:** 23/06/2025

---

### **Resumo**

A poluição do ar é uma das maiores ameaças ambientais à saúde humana. Este estudo investiga a associação quantitativa entre a exposição ao material particulado fino (PM2.5), um indicador chave da qualidade do ar, e as taxas de internações por doenças respiratórias. Utilizando um conjunto de dados de 8 grandes megalópoles, aplicamos um modelo de regressão linear para quantificar essa relação. Os resultados revelam uma correlação positiva e estatisticamente significativa, onde o aumento nos níveis de PM2.5 está diretamente associado a maiores taxas de internações. O modelo de regressão reforçou a urgência de políticas públicas mais rigorosas para o controle da poluição do ar, visando a proteção da saúde pública em escala mundial.

### **Conjunto de Dados**

https://www.kaggle.com/datasets/tfisthis/global-air-quality-and-respiratory-health-outcomes/data

### **Dicionário de Dados: Qualidade do Ar e Saúde**

Esta seção detalha as variáveis presentes no conjunto de dados, fornecendo contexto sobre sua medição e interpretação.

#### **Identificadores**

* `city`
    * **Descrição:** Variável categórica que identifica a cidade onde os dados foram coletados.

* `date`
    * **Descrição:** Data do registro das observações, no formato `AAAA-MM-DD`.

---

#### **Métricas de Qualidade do Ar**

* `aqi` (Índice de Qualidade do Ar)
    * **Descrição:** Métrica adimensional que resume a qualidade geral do ar, calculada com base em múltiplos poluentes.
    * **Interpretação:** Valores mais altos indicam pior qualidade do ar e maior risco à saúde.

* `pm2_5` (Material Particulado Fino)
    * **Descrição:** Concentração de material particulado com diâmetro aerodinâmico de 2.5 micrômetros ou menos.
    * **Unidade:** Microgramas por metro cúbico ($µg/m³$).

* `pm10` (Material Particulado Inalável)
    * **Descrição:** Concentração de material particulado com diâmetro aerodinâmico de 10 micrômetros ou menos.
    * **Unidade:** Microgramas por metro cúbico ($µg/m³$).

* `no2` (Dióxido de Nitrogênio)
    * **Descrição:** Concentração de dióxido de nitrogênio ($NO_2$), um gás poluente gerado pela queima de combustíveis fósseis.
    * **Unidade:** Partes por bilhão (ppb).

* `o3` (Ozônio Troposférico)
    * **Descrição:** Concentração de ozônio ($O_3$) ao nível do solo, um poluente secundário formado na presença de luz solar.
    * **Unidade:** Partes por bilhão (ppb).

---

#### **Fatores Ambientais**

* `temperature`
    * **Descrição:** Temperatura média diária registrada na área de observação.
    * **Unidade:** Graus Celsius (°C).

* `humidity`
    * **Descrição:** Umidade relativa média diária do ar.
    * **Unidade:** Porcentagem (%).

---

#### **Dados de Saúde e Demografia**

* `hospital_admissions`
    * **Descrição:** Contagem diária do número de internações hospitalares por condições respiratórias.

* `population_density`
    * **Descrição:** Variável categórica que classifica a área de observação segundo a densidade populacional.
    * **Categorias:** `Urbana`, `Suburbana`, `Rural`.

* `hospital_capacity`
    * **Descrição:** Número total de leitos hospitalares disponíveis na cidade ou região.

## 1. Instalação das Dependências

A célula abaixo instala todas as bibliotecas Python necessárias para a análise, como Pandas para manipulação de dados, Matplotlib/Seaborn para visualização e NumPy/SciPy para operações numéricas.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy

In [ ]:
# Importação das bibliotecas padrão para análise de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import io
import base64

# Definição do estilo visual mais limpo para os gráficos do notebook
sns.set_style('whitegrid')

In [ ]:
# Carrega o DataFrame e exibe as primeiras linhas.
df = pd.read_csv('/content/air_quality_health_dataset.csv')

# Imprime a legenda da tabela
print("### Amostra Visual do Dataset de Qualidade do Ar")

# Converte o DataFrame para Markdown
print(df.head(10).to_markdown(floatfmt=".2f"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/air_quality_health_dataset.csv'

In [ ]:
# --- 1. Limpeza e Conversão da Coluna 'date' ---

# Converte a coluna para o formato datetime.
# Erros na conversão se tornarão 'NaT' (Not a Time).
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Remove as linhas onde a data é inválida (NaT).
df = df.dropna(subset=['date'])


# --- 2. Filtragem de Datas Futuras ---

# Pega o ano atual para a filtragem. (Ex: 2025)
current_year = pd.Timestamp.now().year

# Cria um novo DataFrame contendo apenas os dados até o ano atual.
df_filtrado = df[df['date'].dt.year <= current_year].copy()


# --- 3. Verificação ---

print("## Verificação da Limpeza e Filtragem de Datas")
print("> Resumo das operações realizadas: a coluna 'date' foi padronizada e as datas futuras foram removidas.")

print("\n### DataFrame Original (`df`)")
print(f"* **Data mais antiga:** `{df['date'].min().date()}`")
print(f"* **Data mais recente:** `{df['date'].max().date()}`")

print("\n### DataFrame Filtrado (`df_filtrado`)")
print(f"* **Data mais antiga:** `{df_filtrado['date'].min().date()}`")
print(f"* **Data mais recente:** `{df_filtrado['date'].max().date()}`")
print(f"* **Número de linhas final:** `{len(df_filtrado)}`")

###**Para essa análise**:

In [ ]:
# --- Remoção da Coluna 'date' ---
df_limpo = df_filtrado.drop('date', axis=1).copy()


# --- Verificação (Saída em Markdown) ---
print("## DataFrame final, sem a coluna 'date'")
print(df_limpo.head().to_markdown(index=False))

print("\n**Colunas restantes no DataFrame:**\n")
# Imprime as colunas como uma lista de itens em um bloco de código
print("```")
for coluna in df_limpo.columns:
    print(f"- {coluna}")
print("```")

# **Análise e Classificação das Variáveis**

Com a exploração inicial, podemos adquirir conhecimento sobre os tipos de dados e classificar cada coluna conforme o tipo de variável que ela representa.

* **Variáveis Qualitativas (Categóricas):** Representam categorias ou rótulos.
    * `city`
    * `population_density`

* **Variáveis Quantitativas Discretas:** Representam contagens ou valores inteiros que não podem ser fracionados.

    * `hospital_admissions`
    * `hospital_capacity`

* **Variáveis Quantitativas Contínuas:** Representam medições que podem assumir qualquer valor dentro de um intervalo.
    * `aqi`
    * `humidity`
    * `pm2_5`
    * `pm10`
    * `no2`
    * `o3`
    * `temperature`

# **Análise das Variáveis Quantitativas Discretas**

Nesta seção, vamos explorar a distribuição e as características das variáveis que representam contagens, que é o caso de `hospital_admissions` e `hospital_capacity`.

##**Extra**: Função para a Plotagem do Gráfico de Frequência

In [ ]:
def gerar_grafico_e_markdown(
    df: pd.DataFrame,
    nome_coluna: str,
    bins: list,
    labels: list,
    titulo: str,
    nome_arquivo: str = None
):
    categorias = pd.cut(df[nome_coluna], bins=bins, labels=labels, right=True, include_lowest=True)
    freq_abs = categorias.value_counts().reindex(labels).fillna(0)

    plt.style.use('seaborn-v0_8-talk')
    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x=freq_abs.index, y=freq_abs.values, palette='viridis', hue=freq_abs.index, legend=False)

    for patch in ax.patches:
        height = patch.get_height()
        ax.annotate(
            text=f'{int(height)}',
            xy=(patch.get_x() + patch.get_width() / 2, height),
            ha='center', va='bottom', xytext=(0, 5), textcoords='offset points'
        )

    plt.title(titulo, fontsize=18, pad=20)
    plt.ylabel('Frequência Absoluta (Contagem)', fontsize=12)
    plt.xlabel('Categoria', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    if nome_arquivo:
        plt.savefig(nome_arquivo, dpi=150, bbox_inches='tight')
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
    plt.close()

    base64_string = base64.b64encode(buffer.getvalue()).decode('utf-8')
    data_uri = f"data:image/png;base64,{base64_string}"

    print(f"## {titulo}\n")
    print(f"![{titulo}]({data_uri})")

##**Extra**: Função para a chamada de Medidas Descritivas

In [ ]:
def gerar_estatisticas_descritivas_md(df: pd.DataFrame, nome_coluna: str):
    # Verifica se a coluna existe no DataFrame
    if nome_coluna not in df.columns:
        print(f"**Erro:** A coluna '{nome_coluna}' não foi encontrada no DataFrame.")
        return

    # --- 1. Cálculo das estatísticas ---
    resumo_series = df[nome_coluna].describe()

    # Adiciona métricas extras ao resumo
    resumo_series['Variância'] = df[nome_coluna].var()
    resumo_series['Assimetria (Skew)'] = df[nome_coluna].skew()
    resumo_series['Curtose (Kurt)'] = df[nome_coluna].kurt()

    # Pega o primeiro valor da moda (pode haver mais de um)
    moda = df[nome_coluna].mode()
    resumo_series['Moda'] = moda.iloc[0] if not moda.empty else None

    # --- 2. Formatação da tabela ---
    tabela_resumo = resumo_series.round(2).reset_index()
    tabela_resumo.columns = ['Medida', 'Valor']

    # Mapeia os nomes técnicos para nomes mais claros
    mapa_nomes = {
        'count': 'Contagem', 'mean': 'Média', 'std': 'Desvio Padrão',
        'min': 'Mínimo', '25%': '1º Quartil (Q1)', '50%': 'Mediana (Q2)',
        '75%': '3º Quartil (Q3)', 'max': 'Máximo', 'Variância': 'Variância',
        'Assimetria (Skew)': 'Assimetria (Skew)', 'Curtose (Kurt)': 'Curtose (Kurt)',
        'Moda': 'Moda'
    }
    tabela_resumo['Medida'] = tabela_resumo['Medida'].map(mapa_nomes)

    # --- 3. Impressão do resultado em Markdown ---
    print(f"## Medidas Descritivas para '{nome_coluna}'\n")
    print(tabela_resumo.to_markdown(index=False))

##**Extra**: Função para gerar o Boxplot das variáveis

In [ ]:
def gerar_boxplot_md(df: pd.DataFrame, nome_coluna: str, titulo: str = None, nome_arquivo: str = None):
    # --- 1. Validação dos dados (silenciosa) ---
    if nome_coluna not in df.columns:
        return
    if not pd.api.types.is_numeric_dtype(df[nome_coluna]):
        return

    # --- 2. Geração e customização do gráfico ---
    plt.style.use('seaborn-v0_8-talk')
    plt.figure(figsize=(8, 10))
    sns.boxplot(y=df[nome_coluna], color=sns.color_palette('viridis')[0])

    if titulo is None:
        titulo = f"Análise de Distribuição (Boxplot) para '{nome_coluna}'"

    plt.title(titulo, fontsize=18, pad=20)
    plt.ylabel("Valores", fontsize=12)
    plt.xlabel(nome_coluna, fontsize=12)
    plt.tight_layout()

    # --- 3. Salvar a imagem no arquivo, com feedback ---
    if nome_arquivo:
        plt.savefig(nome_arquivo, dpi=150, bbox_inches='tight')

    # --- 4. Salvar em memória e converter para Base64 ---
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
    plt.close() # Feche a figura para liberar memória
    base64_string = base64.b64encode(buffer.getvalue()).decode('utf-8')
    data_uri = f"data:image/png;base64,{base64_string}"

    # --- 5. Gerar a saída em Markdown com a imagem incorporada ---
    print(f"## {titulo}\n")
    print(f"![{titulo}]({data_uri})")

##**Análise da Variável `hospital_admissions`**

###**1.1. Tabela de frequência da variável `hospital_admissions`**

In [ ]:
# --- Criação dos Bins e Contagem de Frequência ---
bins = list(np.arange(0, df_limpo['hospital_admissions'].max() + 5, 5))
counts = pd.cut(df['hospital_admissions'], bins=bins, right=False).value_counts().sort_index()


# --- Construção da Tabela de Frequência ---
tabela_admissions = pd.DataFrame({
    'Classe de Internações': counts.index.astype(str),
    'Frequência Absoluta': counts.values,
    'Frequência Relativa (%)': (counts / counts.sum() * 100).map('{:.2f}%'.format)
})


# --- Saída da Tabela em Formato Markdown ---
print("## Tabela de Frequência para 'hospital_admissions'\n")
print(tabela_admissions.to_markdown(index=False))

###**1.2. Gráfico da variável `hospital_admissions`**

In [ ]:
max_admissions = df_limpo['hospital_admissions'].max()
bins_admissions = list(np.arange(0, max_admissions + 5, 5))

labels_admissions = [f'{int(b)} a {int(b+5)-1}' for b in bins_admissions[:-1]]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='hospital_admissions',
    bins=bins_admissions,
    labels=labels_admissions,
    titulo='Distribuição de Frequência da Internacoes Hospitalares',
    nome_arquivo='histograma_hospital_admissions'
)

###**1.3. Medidas Descritivas da variável `hospital_admissions`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='hospital_admissions'
)

###**1.4. Boxplot da variável `hospital_admissions`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='hospital_admissions',
    titulo='Distribuição de Internações Hospitalares',
    nome_arquivo='distribuicao_internacoes'
)

##**Análise da Variável `hospital_capacity`**

###**2.1. Tabela de frequência da variável `hospital_capacity`**

In [ ]:
bins_hcap = list(np.arange(0, df_limpo['hospital_capacity'].max() + 500, 500))
counts = pd.cut(df['hospital_capacity'], bins=bins_hcap, right=False).value_counts().sort_index()


# --- Construção da Tabela de Frequência ---
tabela_capacity = pd.DataFrame({
    'Classe de Capacidade': counts.index.astype(str),
    'Frequência Absoluta': counts.values,
    'Frequência Relativa (%)': (counts / counts.sum() * 100).map('{:.2f}%'.format)
})


# --- Saída da Tabela em Formato Markdown ---
print("## Tabela de Frequência para 'hospital_capacity'\n")
print(tabela_capacity.to_markdown(index=False))

###**2.2. Gráfico da variável `hospital_capacity`**

In [ ]:
# Passo 1: Definir os parâmetros para a capacidade hospitalar
max_capacity = df_limpo['hospital_capacity'].max()
bins_capacity = list(np.arange(0, max_capacity + 500, 500))

labels_capacity = [f'{int(b)} a {int(b+500)-1}' for b in bins_capacity[:-1]]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='hospital_capacity',
    bins=bins_capacity,
    labels=labels_capacity,
    titulo='Distribuição de Frequência da Capacidade Hospitalar'
)

###**2.3. Medidas Descritivas da variável `hospital_capacity`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='hospital_capacity'
)

###**2.4. Boxplot da variável `hospital_capacity`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='hospital_capacity',
    titulo='Distribuição de Capacidade Hospitalar'
)

#**Análise das Variáveis Quantitativas Contínuas**

Nesta seção, vamos explorar a distribuição e as características das variáveis que representam medições, que é o caso de `aqi`, `humidity`, `pm2_5`, `pm10`, `no2`, `o3`, `temperature` .

##**Análise da Variável `aqi` (Índice de Qualidade do Ar)**


Essa variável diz respeito ao Índice de Qualidade do Ar (AQI), que é um número usado para transmitir a qualidade do ar pelo governo ao público em geral. A qualidade do ar se deteriora com o aumento da concentração de poluentes, e o Índice de Qualidade do Ar representa a gravidade da poluição para as pessoas comuns.

Abaixo, temos a classificação padrão do AQI.

| Nível de Risco (Faixa de AQI)        | Significado                                                                                             | Recomendação de Saúde                                                                |
| :----------------------------------- | :------------------------------------------------------------------------------------------------------ | :----------------------------------------------------------------------------------- |
| **Boa (0 - 50)** | A qualidade do ar é boa e representa pouco ou nenhum risco.                                             | **Todos:** Podem desfrutar de atividades ao ar livre.                                |
| **Moderada (51 - 100)** | A qualidade do ar é aceitável, mas pode ser uma preocupação para pessoas muito sensíveis à poluição.      | **Grupos Sensíveis:** Devem planejar atividades intensas para quando a qualidade do ar estiver melhor. |
| **Prejudicial a Grupos Sensíveis (101 - 150)** | A qualidade do ar é prejudicial para grupos sensíveis, incluindo adultos ativos, idosos e crianças. | **Grupos Sensíveis:** Devem reduzir ou reagendar atividades físicas intensas ao ar livre. |
| **Prejudicial à Saúde (151 - 200)** | A qualidade do ar é prejudicial para todos, especialmente para pessoas com doenças cardíacas ou pulmonares. | **Todos:** Devem evitar atividades físicas intensas ao ar livre.                      |
| **Muito Prejudicial à Saúde (201 - 300)** | A qualidade do ar é muito prejudicial para todos, especialmente para pessoas com doenças cardíacas ou pulmonares. | **Todos:** Devem evitar atividades físicas ao ar livre.                               |
| **Perigosa (301 - 500)** | A qualidade do ar é perigosa para todos.                                                                | **Todos:** Devem evitar qualquer atividade ao ar livre.                              |

*Fonte: Imagem em [K9 Mask](https://pt.k9mask.com/Blogs/not%C3%ADcia/No%C3%A7%C3%B5es-b%C3%A1sicas-sobre-o-%C3%ADndice-de-qualidade-do-ar-aqi-para-o-seu-c%C3%A3o)*

###**Tabela de frequência da variável `aqi`**

In [ ]:
limites_intervalos = [-1, 50, 100, 150, 200, 300, np.inf]
rotulos = [
    'Boa (0-50)',
    'Moderada (51-100)',
    'Prejudicial a Grupos Sensíveis (101-150)',
    'Prejudicial à Saúde (151-200)',
    'Muito Prejudicial à Saúde (201-300)',
    'Perigosa (301+)'
]

contagens = pd.cut(df_limpo['aqi'], bins=limites_intervalos, labels=rotulos).value_counts().reindex(rotulos).fillna(0)

tabela_aqi = pd.DataFrame({
    'Categoria AQI': contagens.index,
    'Frequência Absoluta (Dias)': contagens.values.astype(int),
    'Frequência Relativa (%)': (contagens / contagens.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'aqi':")
print(f"\n{tabela_aqi.to_string(index=False)}")

###**Gráfico da variável `aqi`**

In [ ]:
bins_aqi = [-1, 50, 100, 150, 200, 300, np.inf]

labels_aqi = [
    'Boa (0-50)',
    'Moderada (51-100)',
    'Prejudicial a Grupos Sensíveis (101-150)',
    'Prejudicial à Saúde (151-200)',
    'Muito Prejudicial à Saúde (201-300)',
    'Perigosa (301+)'
]
# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='aqi',
    bins=bins_aqi,
    labels=labels_aqi,
    titulo='Distribuição de Frequência do Indice de Qualidade do Ar'
)

###**Medidas Descritivas da variável `aqi`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='aqi'
)

###**Boxplot da variável `aqi`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='aqi',
    titulo='Distribuição do Indíce de Qualidade do Ar'
)

##**Análise da Variável `humidity`**


Para contextualizar a variável de umidade, vamos adotar uma classificação padrão, frequentemente utilizada por órgãos de saúde e meteorologia no Brasil. Esta abordagem nos permite transformar a variável numérica `humidity` em uma categoria de fácil interpretação.

A tabela abaixo resume as faixas que serão utilizadas em nossa análise:

| Nível de Umidade (%)  | Classificação                | Impacto na Saúde e Conforto                                        |
|:----------------------|:-----------------------------|:-------------------------------------------------------------------|
| **Abaixo de 30%** | `Estado de Atenção`          | Ar muito seco, prejudicial à saúde (pode causar problemas respiratórios). |
| **Entre 30% e 60%** | `Faixa Ideal / Confortável`  | Nível considerado ideal para o bem-estar e a saúde humana.         |
| **Acima de 60%** | `Estado de Alerta`           | Ar úmido, pode causar desconforto e favorecer a proliferação de mofo. |

Com base nesta classificação, criaremos uma nova coluna categórica no DataFrame para facilitar as análises subsequentes.

###**Tabela de frequência da variável `humidity`**

In [ ]:
bins = [-1, 30, 60, df_limpo['humidity'].max()]
labels = [
    'Baixa (0-30%)',
    'Ideal (31-60%)',
    'Alta (61% e acima)'
]

counts = pd.cut(df['humidity'], bins=bins, labels=labels).value_counts().reindex(labels).fillna(0)

tabela_humidity = pd.DataFrame({
    'Categoria de Umidade': counts.index,
    'Frequência Absoluta (Dias)': counts.values.astype(int),
    'Frequência Relativa (%)': (counts / counts.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'humidity' por Categoria de Conforto:")
print(f"\n{tabela_humidity.to_string(index=False)}")

###**Gráfico da variável `humidity`**

In [ ]:
bins_humidity = [-1, 30, 60, df['humidity'].max()]
labels_humidity = [
    'Baixa (0-30%)',
    'Ideal (31-60%)',
    'Alta (61% e acima)'
]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='humidity',
    bins=bins_humidity,
    labels=labels_humidity,
    titulo='Distribuição de Frequência da Umidade do Ar'
)

###**Medidas Descritivas da variável `humidity`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='humidity'
)

###**Boxplot da variável `humidity`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='humidity',
    titulo='Distribuição do Umidade do Ar'
)

##**Análise da Variável `pm2_5`**

O termo PM2.5 refere-se a material particulado fino (em inglês, Particulate Matter), que são partículas poluentes extremamente pequenas suspensas no ar, com um diâmetro aerodinâmico de 2.5 micrômetros (µm) ou menos.

Classificação da Qualidade do Ar baseada em PM2.5 (Padrão US-EPA)

Aqui está uma tabela que mostra como os níveis de PM2.5 são classificados, suas cores correspondentes e o que significam para a saúde:

| Categoria | Níveis de PM2.5 (média de 24h) | Cor do Índice | Recomendações de Saúde |
|:---|:---|:---|:---|
| **Boa** | 0 - 12.0 µg/m³ | **Verde** | A qualidade do ar é excelente e a poluição representa pouco ou nenhum risco. |
| **Moderada** | 12.1 - 35.4 µg/m³ | **Amarela** | Qualidade do ar aceitável. Pessoas muito sensíveis devem considerar reduzir atividades externas intensas. |
| **Insalubre para<br>Grupos Sensíveis** | 35.5 - 55.4 µg/m³ | **Laranja** | Pessoas com doenças cardíacas ou pulmonares, idosos e crianças correm maior risco. Devem limitar atividades externas prolongadas. |
| **Insalubre** | 55.5 - 150.4 µg/m³ | **Vermelha** | Todos podem começar a sentir efeitos na saúde. Grupos sensíveis podem ter efeitos mais sérios. Limite as atividades externas. |
| **Muito Insalubre** | 150.5 - 250.4 µg/m³ | **Roxa** | Alerta de saúde. O risco de efeitos na saúde é aumentado para todos. Evite todas as atividades externas. |
| **Perigosa** | ≥ 250.5 µg/m³ | **Marrom** | Condição de emergência. Toda a população tem maior probabilidade de ser afetada. Permaneça em ambientes internos. |


*Imagem em [Research Gate](https://www.researchgate.net/figure/PM25-levels-source-wwwepagov_fig5_353328349)*

###**Tabela de frequência da variável `pm2_5`**

In [ ]:
limites_intervalos = [-1, 12.1, 35.5, 55.5, 150.5, 250.5, np.inf]
rotulos = [
    'Boa (0-12.0)',
    'Moderada (12.1-35.4)',
    'Prejudicial a Grupos Sensíveis (35.5-55.4)',
    'Prejudicial à Saúde (55.5-150.4)',
    'Muito Prejudicial à Saúde (201-300)',
    'Perigosa (301+)'
]

contagens = pd.cut(df_limpo['pm2_5'], bins=limites_intervalos, labels=rotulos).value_counts().reindex(rotulos).fillna(0)

tabela_aqi = pd.DataFrame({
    'Categoria PM2_5': contagens.index,
    'Frequência Absoluta (Dias)': contagens.values.astype(int),
    'Frequência Relativa (%)': (contagens / contagens.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'pm2_5':")
print(f"\n{tabela_aqi.to_string(index=False)}")

###**Gráfico da variável `pm2_5`**

In [ ]:
bins_pm2_5 = [-1, 12.1, 35.5, 55.5, 150.5, 250.5, np.inf]
labels_pm2_5 = [
    'Boa (0-12.0)',
    'Moderada (12.1-35.4)',
    'Prejudicial a Grupos Sensíveis (35.5-55.4)',
    'Prejudicial à Saúde (55.5-150.4)',
    'Muito Prejudicial à Saúde (150.5-250.4)',
    'Perigosa (250.5+)'
]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='pm2_5',
    bins=bins_pm2_5,
    labels=labels_pm2_5,
    titulo='Distribuição de Material Particulado Fino no Ar',
    nome_arquivo='histograma_pm25.png'
)

###**Medidas Descritivas da variável `pm2_5`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='pm2_5'
)

###**Boxplot da variável `pm2_5`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='pm2_5',
    titulo='Distribuição do Indíce de Partículas Finas no Ar',
    nome_arquivo='distribuicao_pm25'
)

##**Análise da Variável `pm10`**


PM10 (ou Material Particulado 10) refere-se a partículas inaláveis com diâmetro aerodinâmico de 10 micrômetros (µm) ou menos. Esta categoria é mais abrangente que a do PM2.5 e inclui tanto as partículas finas (PM2.5) quanto as partículas grossas (com diâmetro entre 2.5 e 10 µm)

Aqui está uma tabela que mostra como os níveis de PM2.5 são classificados, suas cores correspondentes e o que significam para a saúde:

| Níveis de PM₁₀ (µg/m³) | Cor | Classificação |
|:---|:---|:---|
| 0 - 50 | **Verde** | Boa |
| 51 - 100 | **Azul** | Moderada |
| 101 - 199 | **Amarelo** | Insalubre |
| 200 - 300 | **Vermelho** | Muito Insalubre |
| > 300 | **Preto** | Perigosa |

*Imagem em [Research Gate](https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.researchgate.net%2Ffigure%2FPM10-range-value-based-on-Air-Pollution-Standards-Index-13_tbl2_342802805&psig=AOvVaw3Nl7ySacvJ0Lq6HACR-k2i&ust=1750795772386000&source=images&cd=vfe&opi=89978449&ved=0CBUQjRxqFwoTCIiwkdWsiI4DFQAAAAAdAAAAABAE)*

###**Tabela de frequência da variável 'pm10'**

In [ ]:
limites_intervalos = [-1, 51, 101, 200, 301, np.inf]
rotulos = [
    'Boa (0-50)',
    'Moderada (51-100)',
    'Insalubre (101-199)',
    'Muito Insalubre (200-300)',
    'Perigosa (301+)'
]

contagens = pd.cut(df_limpo['pm10'], bins=limites_intervalos, labels=rotulos).value_counts().reindex(rotulos).fillna(0)

tabela_aqi = pd.DataFrame({
    'Categoria PM10': contagens.index,
    'Frequência Absoluta (Dias)': contagens.values.astype(int),
    'Frequência Relativa (%)': (contagens / contagens.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'pm10':")
print(f"\n{tabela_aqi.to_string(index=False)}")

###**Gráfico da variável 'pm10'**

In [ ]:
bins_pm10 = [-1, 51, 101, 200, 301, np.inf]
labels_pm10 = [
    'Boa (0-50)',
    'Moderada (51-100)',
    'Insalubre (101-199)',
    'Muito Insalubre (200-300)',
    'Perigosa (301+)'
]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='pm10',
    bins=bins_pm10,
    labels=labels_pm10,
    titulo='Distribuição de Material Particulado Inalável no Ar'
)

###**Medidas Descritivas da variável 'pm10'**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='pm10'
)

###**Boxplot da variável 'pm10'**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='pm10',
    titulo='Distribuição do Indíce de Partículas Menos Finas no Ar'
)

##**Análise da Variável `no2`**


O Dióxido de Nitrogênio é um gás de cor castanho-avermelhada, com um odor forte e irritante. Ele é um dos principais gases do grupo dos óxidos de nitrogênio (NOx) e é um poluente altamente reativo.

A classificação do NO₂ geralmente é baseada na sua concentração média ao longo de uma hora, pois seus efeitos agudos são bastante significativos. A unidade de medida mais comum para gases como o NO₂ no Índice de Qualidade do Ar (IQA) é ppb (partes por bilhão).
Padrão de Referência (baseado no IQA da US-EPA)

| Categoria | Níveis de NO₂ (média de 1h) | Cor do Índice | Recomendações de Saúde |
|:---|:---|:---|:---|
| **Boa** | 0 - 53 ppb | **Verde** | A qualidade do ar é excelente. |
| **Moderada** | 54 - 100 ppb | **Amarela** | Qualidade do ar aceitável. Pessoas muito sensíveis a NO₂ (como asmáticos) devem considerar limitar atividades externas intensas. |
| **Insalubre para<br>Grupos Sensíveis** | 101 - 360 ppb | **Laranja** | Pessoas com doenças respiratórias (especialmente asma) correm maior risco de problemas respiratórios. |
| **Insalubre** | 361 - 649 ppb | **Vermelha** | Aumento da probabilidade de problemas respiratórios em grupos sensíveis. A população em geral pode começar a sentir efeitos. |
| **Muito Insalubre** | 650 - 1249 ppb | **Roxa** | Todos podem sentir efeitos respiratórios mais sérios. Evite atividades externas prolongadas. |
| **Perigosa** | ≥ 1250 ppb | **Marrom** | Condição de emergência para a saúde. Permaneça em ambientes internos. |

###**Tabela de frequência da variável `no2`**

In [ ]:
limites_intervalos = [-1, 54, 101, 361, 650, 1250, np.inf]
rotulos = [
    'Boa (0-53)',
    'Moderada (54-100)',
    'Insalubre para Grupos Sensíveis (101-360)',
    'Insalubre (361-649)',
    'Muito Insalubre (650-1249)',
    'Perigosa (1250+)'
]

contagens = pd.cut(df_limpo['no2'], bins=limites_intervalos, labels=rotulos).value_counts().reindex(rotulos).fillna(0)

tabela_aqi = pd.DataFrame({
    'Categoria NO2': contagens.index,
    'Frequência Absoluta (Dias)': contagens.values.astype(int),
    'Frequência Relativa (%)': (contagens / contagens.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'no2':")
print(f"\n{tabela_aqi.to_string(index=False)}")

###**Gráfico da variável `no2`**

In [ ]:
bins_no2 = [-1, 54, 101, 361, 650, 1250, np.inf]
labels_no2 = [
    'Boa (0-53)',
    'Moderada (54-100)',
    'Insalubre para Grupos Sensíveis (101-360)',
    'Insalubre (361-649)',
    'Muito Insalubre (650-1249)',
    'Perigosa (1250+)'
]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='no2',
    bins=bins_no2,
    labels=labels_no2,
    titulo='Distribuição de Dioxido de Nitrogenio no Ar'
)

###**Medidas Descritivas da variável `no2`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='no2'
)

###**Boxplot da variável `no2`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='no2',
    titulo='Distribuição do Indíce de Dióxido de Nitrogênio no Ar'
)

##**Análise da Variável `o3`**


Resumo sobre Ozônio Troposférico (O₃)
O Ozônio "Bom" vs. o Ozônio "Ruim"

É crucial entender que existem dois tipos de ozônio:

    Ozônio Estratosférico (Bom): Fica na alta atmosfera (estratosfera)
    e forma a "camada de ozônio", que nos protege da radiação
    ultravioleta (UV) do sol. Este ozônio é benéfico e essencial para
    a vida na Terra. Ozônio Troposférico (Ruim): É o ozônio que se forma perto do solo, na camada de ar que respiramos
    (troposfera). Este é um poluente atmosférico nocivo, principal componente da névoa fotoquímica (smog).
    Quando falamos de ozônio como poluente, estamos nos referindo a este tipo.

A classificação para o ozônio geralmente se baseia na sua concentração média ao longo de 8 horas, pois os efeitos da exposição prolongada durante o dia são muito relevantes. A unidade padrão para o ozônio no IQA é ppb (partes por bilhão).

| Categoria | Níveis de O₃ (média de 8h) | Cor do Índice | Recomendações de Saúde |
|:---|:---|:---|:---|
| **Boa** | 0 - 54 ppb | **Verde** | A qualidade do ar é excelente. |
| **Moderada** | 55 - 70 ppb | **Amarela** | Pessoas extraordinariamente sensíveis ao ozônio devem considerar limitar atividades externas prolongadas e intensas. |
| **Insalubre para<br>Grupos Sensíveis** | 71 - 85 ppb | **Laranja** | Crianças, idosos e pessoas com doenças pulmonares (como asma) devem limitar atividades externas prolongadas. |
| **Insalubre** | 86 - 105 ppb | **Vermelha** | Crianças, idosos e pessoas com doenças pulmonares devem evitar atividades externas. Todos os outros devem limitar esforço prolongado ao ar livre. |
| **Muito Insalubre** | 106 - 200 ppb | **Roxa** | Grupos sensíveis devem evitar qualquer atividade externa. Todos os outros devem limitar muito as atividades ao ar livre. |
| **Perigosa** | ≥ 201 ppb | **Marrom** | Alerta de saúde. Todos devem evitar qualquer esforço ao ar livre. Permaneça em ambientes internos. |

###**Tabela de frequência da variável `o3`**

In [ ]:
limites_intervalos = [-1, 55, 71, 86, 106, 201, np.inf]
rotulos = [
    'Boa (0-54)',
    'Moderada (55-70)',
    'Insalubre para Grupos Sensíveis (71-85)',
    'Insalubre (86-105)',
    'Muito Insalubre (106-200)',
    'Perigosa (201+)'
]

contagens = pd.cut(df_limpo['o3'], bins=limites_intervalos, labels=rotulos).value_counts().reindex(rotulos).fillna(0)

tabela_aqi = pd.DataFrame({
    'Categoria o3': contagens.index,
    'Frequência Absoluta (Dias)': contagens.values.astype(int),
    'Frequência Relativa (%)': (contagens / contagens.sum() * 100).map('{:.2f}%'.format)
})

print("Tabela de Frequência para 'o3':")
print(f"\n{tabela_aqi.to_string(index=False)}")

###**Gráfico da variável `o3`**

In [ ]:
bins_o3 = [-1, 55, 71, 86, 106, 201, np.inf]
labels_o3 = [
    'Boa (0-54)',
    'Moderada (55-70)',
    'Insalubre para Grupos Sensíveis (71-85)',
    'Insalubre (86-105)',
    'Muito Insalubre (106-200)',
    'Perigosa (201+)'
]

# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='o3',
    bins=bins_o3,
    labels=labels_o3,
    titulo='Distribuição de Dioxido de Nitrogenio no Ar'
)

###**Medidas Descritivas da variável `o3`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='o3'
)

###**Boxplot da variável `o3`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='o3',
    titulo='Distribuição do Indíce de Ozônio no Ar'
)

##**Análise da Variável `temperature`**


###**Tabela de frequência da variável `temperature`**

In [ ]:
# --- Criação dos Bins e Contagem de Frequência ---
bins = list(np.arange(df_limpo['temperature'].min(), df_limpo['temperature'].max() + 5, 5))
counts = pd.cut(df['temperature'], bins=bins, right=False).value_counts().sort_index()


# --- Construção da Tabela de Frequência ---
tabela_admissions = pd.DataFrame({
    'Classe de Temperatura': counts.index.astype(str),
    'Frequência Absoluta': counts.values,
    'Frequência Relativa (%)': (counts / counts.sum() * 100).map('{:.2f}%'.format)
})


# --- Saída da Tabela em Formato Markdown ---
print("## Tabela de Frequência para Temperatura\n")
print(tabela_admissions.to_markdown(index=False))

###**Gráfico da variável `temperature`**

In [ ]:
max_temp = df_limpo['temperature'].max()
min_temp = df_limpo['temperature'].min()

min_temp_arredondado = np.floor(min_temp / 5) * 5

bins_temp = list(np.arange(min_temp_arredondado, max_temp + 5, 5))

labels_temp = [f'[{int(b)}, {int(b+5)})' for b in bins_temp[:-1]]
# Passo 2: Chamar a nova função para gerar o gráfico e o código Markdown
gerar_grafico_e_markdown(
    df=df_limpo,
    nome_coluna='temperature',
    bins=bins_temp,
    labels=labels_temp,
    titulo='Distribuição de Temperatura'
)

###**Medidas Descritivas da variável `temperature`**

In [ ]:
gerar_estatisticas_descritivas_md(
    df=df_limpo,
    nome_coluna='temperature'
)

###**Boxplot da variável `temperature`**

In [ ]:
gerar_boxplot_md(
    df=df_limpo,
    nome_coluna='temperature',
    titulo='Distribuição da Temperatura'
)

#**Análise das Variáveis Qualitativas**

Nesta seção, vamos explorar a distribuição e as características das variáveis que representam categorias ou rótulos, que é o caso de `city` e `population_density`.



## **Análise da Variável `city`**

In [ ]:
plt.style.use('seaborn-v0_8-talk')
plt.figure(figsize=(16, 9))

# Contar a frequência de cada cidade
freq_cidades = df['city'].value_counts()

# Criar o gráfico de barras
ax = sns.barplot(
    x=freq_cidades.index,
    y=freq_cidades.values,
    palette='viridis',
    hue=freq_cidades.index,
    dodge=False,
    legend=False
)

# Adicionar os rótulos com a contagem exata em cima de cada barra
for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(
        text=f'{int(height)}',
        xy=(patch.get_x() + patch.get_width() / 2, height),
        ha='center',
        va='bottom',
        xytext=(0, 5), # Desloca o texto um pouco para cima da barra
        textcoords='offset points',
        fontsize=10 # Ajuste o tamanho da fonte para os rótulos
    )

# --- Customização dos Títulos e Eixos ---
plt.title('Frequência de Observações por Cidade', fontsize=20, pad=20)
plt.xlabel('Cidade', fontsize=14)
plt.ylabel('Frequência (Contagem)', fontsize=14)
plt.xticks(rotation=45, ha='right') # Rotacionar nomes das cidades para não sobrepor
plt.legend([],[], frameon=False) # Ocultar a legenda desnecessária
plt.tight_layout() # Ajusta automaticamente os parâmetros do subplot para que caibam na figura.

plt.savefig('city.png', format='png', bbox_inches='tight')

# Salvar o gráfico em um buffer de memória
buffer = io.BytesIO()
plt.savefig(buffer, format='png', bbox_inches='tight')
buffer.seek(0)
image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
plt.close() # Fecha a figura para liberar memória

# Gerar o output no formato desejado
output = f"""
## Distribuição de Frequência de Observações por Cidade

![Frequência de Observações por Cidade](data:image/png;base64,{image_base64})
"""

print(output)

# **Análise de Correlação entre as Variáveis**


A análise de correlação é uma etapa fundamental para entender a relação entre as diferentes variáveis quantitativas do nosso conjunto de dados. A matriz de correlação abaixo nos ajuda a identificar quais variáveis se movem juntas e a força dessa relação. Valores próximos de **+1** indicam uma forte correlação positiva (quando uma sobe, a outra também sobe), enquanto valores próximos de **-1** indicam uma forte correlação negativa (quando uma sobe, a outra desce). Valores perto de **0** sugerem ausência de correlação linear.

Para esta análise, vamos focar nas variáveis numéricas para visualizar suas interconexões através de um *heatmap* (mapa de calor), o que facilita a interpretação visual dos coeficientes de correlação.


In [ ]:
# --- Bloco de Análise e Geração de Imagem ---
df_numeric = df_limpo.select_dtypes(include=np.number)
correlation_matrix = df_numeric.corr()

labels_curtos = {
    'aqi': 'AQI', 'pm2_5': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2',
    'o3': 'O3', 'temperature': 'Temperatura', 'humidity': 'Umidade',
    'hospital_admissions': 'Internações', 'hospital_capacity': 'Capacidade Hosp.'
}

correlation_matrix_curta = correlation_matrix.rename(columns=labels_curtos, index=labels_curtos)

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_curta, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Matriz de Correlação', fontsize=18, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()

# --- Adição para salvar a imagem em um arquivo ---
# Defina o nome do arquivo aqui, por exemplo:
nome_arquivo_correlacao = 'matriz_correlacao.png'
try:
    plt.savefig(nome_arquivo_correlacao, dpi=150, bbox_inches='tight')
    print(f"Matriz de correlação salva como: {nome_arquivo_correlacao}")
except Exception as e:
    print(f"Erro ao salvar a matriz de correlação em {nome_arquivo_correlacao}: {e}")

# --- Salvar em memória e converter para Base64 ---
buffer = io.BytesIO()
plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
plt.close()
base64_string = base64.b64encode(buffer.getvalue()).decode('utf-8')

# --- Geração do Output em Markdown ---
codigo_markdown_para_output = f"""
### Gráfico da Matriz de Correlação

O mapa de calor a seguir visualiza a correlação entre as variáveis numéricas do estudo.

![Matriz de Correlação](data:image/png;base64,{base64_string})
"""
print(codigo_markdown_para_output)

#**Regressão Linear Simples**

O gráfico de regressão a seguir visualiza a relação entre a concentração de PM2.5 e o número diário de internações hospitalares. A linha vermelha representa a tendência central dos dados, enquanto os pontos azuis mostram a dispersão das observações diárias. As métricas estatísticas (equação da reta, R² e p-valor) são exibidas no canto superior esquerdo para quantificar a força e a significância da relação.

In [ ]:
from scipy.stats import linregress

# Carregar os dados
df = df_limpo

# --- Análise de Regressão para PM2.5 vs Internações ---

# Definir as variáveis X (preditora) e Y (resposta)
x_var = df['pm2_5']
y_var = df['hospital_admissions']

# Gerar o gráfico de dispersão com a linha de regressão
plt.figure(figsize=(12, 7))
sns.regplot(x=x_var, y=y_var,
            line_kws={"color": "red", "linewidth": 3},
            scatter_kws={"alpha": 0.3, "color": "green"})

# Calcular os resultados da regressão para obter a equação e o R²
slope, intercept, r_value, p_value, std_err = linregress(x_var, y_var)
r_squared = r_value**2
equacao = f'y = {slope:.4f}x + {intercept:.4f}'
r2_texto = f'R² = {r_squared:.4f}'

# Adicionar a equação e o R² ao gráfico
plt.text(0.05, 0.95, equacao, transform=plt.gca().transAxes, fontsize=14, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))
plt.text(0.05, 0.88, r2_texto, transform=plt.gca().transAxes, fontsize=14, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

plt.title('Análise de Regressão Linear: PM2.5 vs. Internações Hospitalares', fontsize=18, pad=20)
plt.xlabel('Concentração de Partículas Finas PM2.5 (µg/m³)', fontsize=14)
plt.ylabel('Número de Internações Diárias', fontsize=14)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()

# --- Adição para salvar a imagem em um arquivo ---
# Define the filename for the saved plot
nome_arquivo_regressao = 'regressao_pm25_internacoes.png'
try:
    plt.savefig(nome_arquivo_regressao, dpi=150, bbox_inches='tight')
    print(f"Gráfico de regressão salvo como: {nome_arquivo_regressao}")
except Exception as e:
    print(f"Erro ao salvar o gráfico de regressão em {nome_arquivo_regressao}: {e}")

# --- Salvar em memória e converter para Base64 ---
buffer = io.BytesIO()
plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
plt.close()
base64_string = base64.b64encode(buffer.getvalue()).decode('utf-8')

# --- Geração do Output em Markdown ---
codigo_markdown_para_output = f"""
### Gráfico da Análise de Regressão Linear

O gráfico de dispersão a seguir mostra a relação entre a concentração de PM2.5 e o número de internações hospitalares, com a linha de regressão linear ajustada, a equação da reta e o coeficiente de determinação (R²).

![Análise de Regressão Linear](data:image/png;base64,{base64_string})
"""
print(codigo_markdown_para_output)

### Análise de Resíduos do Modelo

Após construir um modelo de regressão, a análise de resíduos é uma etapa de diagnóstico crucial para verificar sua validade e confiabilidade. Os resíduos representam a diferença entre os valores reais (as internações observadas) e os valores que o nosso modelo previu. Essencialmente, eles são os 'erros' do modelo para cada ponto de dado. Para que o nosso modelo de regressão linear seja considerado adequado, esses erros não devem seguir nenhum padrão; eles precisam se comportar de forma aleatória, indicando que a linha de regressão capturou bem a tendência principal dos dados.

Para verificar essa aleatoriedade e validar as suposições do modelo, vamos plotar os resíduos em gráficos de diagnóstico. O primeiro analisará a dispersão dos resíduos em relação aos valores previstos, ajudando a identificar tendências ou problemas como a falta de linearidade. O segundo, um gráfico Q-Q (Quantil-Quantil), nos permitirá avaliar se os erros seguem uma distribuição normal. Essa inspeção visual é fundamental para confirmar se as suposições do nosso modelo foram atendidas, garantindo a robustez das nossas conclusões.

In [ ]:
from scipy import stats

# Análise e cálculo dos resíduos

try:
    df_limpo
except NameError:
    print("df_limpo not found. Creating a dummy DataFrame for demonstration.")
    data = {
        'pm2_5': np.random.rand(100) * 50 + 10,
        'hospital_admissions': np.random.rand(100) * 100 + 50 + (np.random.rand(100) * 50 + 10) * 0.5
    }
    df_limpo = pd.DataFrame(data)

x_var = df_limpo['pm2_5']
y_var = df_limpo['hospital_admissions']
slope, intercept, _, _, _ = stats.linregress(x_var, y_var)
y_pred = slope * x_var + intercept
residuos = y_var - y_pred

# Criação dos gráficos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(x=y_pred, y=residuos, alpha=0.6, ax=ax1, s=50, edgecolor='k', linewidth=0.5)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_title('Gráfico de Resíduos vs. Valores Previstos', fontsize=15, pad=10)
ax1.set_xlabel('Valores Previstos de Internações', fontsize=12)
ax1.set_ylabel('Resíduos', fontsize=12)
ax1.grid(linestyle='--', alpha=0.6)

stats.probplot(residuos, dist="norm", plot=ax2)
ax2.get_lines()[0].set_markerfacecolor('#1f77b4')
ax2.get_lines()[1].set_linewidth(2)
ax2.set_title('Gráfico Q-Q de Normalidade dos Resíduos', fontsize=15, pad=10)
ax2.set_xlabel('Quantis Teóricos da Normal', fontsize=12)
ax2.set_ylabel('Quantis dos Resíduos', fontsize=12)
ax2.grid(linestyle='--', alpha=0.6)
fig.suptitle('Análise de Resíduos para a Regressão de PM2.5 vs. Internações', fontsize=18, y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.96])

# --- Adição para salvar a imagem em um arquivo ---
nome_arquivo_residuos = 'analise_residuos_regressao.png'
try:
    plt.savefig(nome_arquivo_residuos, dpi=150, bbox_inches='tight')
    print(f"Gráfico de análise de resíduos salvo como: {nome_arquivo_residuos}")
except Exception as e:
    print(f"Erro ao salvar o gráfico de resíduos em {nome_arquivo_residuos}: {e}")


# --- Salvar em memória e converter para Base64 ---
buffer = io.BytesIO()
plt.savefig(buffer, format='png', dpi=150, bbox_inches='tight')
plt.close()
base64_string = base64.b64encode(buffer.getvalue()).decode('utf-8')

# --- Bloco de Geração do Output ---
codigo_markdown_para_output = f"""
### Análise de Resíduos: Regressão PM2.5 vs. Internações

Os gráficos abaixo são essenciais para validar a adequação do modelo de regressão linear. O *Gráfico de Resíduos* (esquerda) mostra se há padrões nos erros do modelo, enquanto o *Gráfico Q-Q* (direita) verifica se os erros seguem uma distribuição normal.

![Análise de Resíduos da Regressão](data:image/png;base64,{base64_string})
"""
print(codigo_markdown_para_output)

#**Conclusão**

Esta análise quantifica de forma clara a ligação prejudicial entre a poluição do ar por partículas finas (PM2,5) e as internações por doenças respiratórias em escala global. Os dados estatísticos não apenas confirmam uma correlação forte e positiva, mas também demonstram sua alta relevância. Fica evidente, portanto, que a qualidade do ar é um fator fundamental para a saúde respiratória da população.

Apesar das limitações do modelo de análise utilizado, a mensagem é clara: a poluição do ar é uma crise de saúde pública que exige ação imediata. As conclusões deste trabalho reforçam a importância de implementar e fiscalizar políticas ambientais mais rígidas, como o controle de emissões industriais e veiculares, a transição para energias renováveis e o planejamento urbano sustentável. Essas ações são essenciais para diminuir os impactos da má qualidade do ar, reduzir o sofrimento humano e, fundamentalmente, salvar vidas.